# Governed Bundle Workflow

Purpose: prove the first product-facing workflow end to end through the CLI-backed governed bundle export.

Mode: proof

Related docs:
- `docs/plans/0001_successor_roadmap.md`
- `docs/plans/0004_phase10_governed_bundle_shape.md`
- `docs/adr/0010-choose-cli-driven-governed-bundle-export-as-the-first-product-facing-workflow.md`


## Phase 1

Purpose: set up one real source file and a deterministic extractor stand-in.

Input -> output: `source_text_file -> extracted_candidate_batch`

Acceptance criteria:
1. The workflow uses a concrete source file.
2. CLI extraction persists a candidate and proposal batch.
3. Execution remains live except for the deterministic extractor stand-in.

status: `proven`

execution_mode: `fixture`

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not locate onto-canon6 repo root from notebook cwd")

for candidate_path in (PROJECT_ROOT / "src", PROJECT_ROOT.parent / "llm_client"):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

import contextlib
import io
import json
from tempfile import TemporaryDirectory

from onto_canon6 import cli as cli_module
from onto_canon6.pipeline import CandidateAssertionImport, EvidenceSpan, ProfileRef, ReviewService, SourceArtifactRef


In [2]:
class DeterministicTextExtractionService:
    """Provide a fixture-shaped extraction stand-in while the CLI flow stays real."""

    def __init__(self, *, review_service: ReviewService) -> None:
        self._review_service = review_service

    def extract_and_submit(
        self,
        *,
        source_text: str,
        profile_id: str,
        profile_version: str,
        submitted_by: str,
        source_ref: str,
        source_kind: str = 'text_file',
        source_label: str | None = None,
        source_metadata: dict[str, object] | None = None,
    ) -> tuple[object, ...]:
        del source_metadata
        phrase = 'aligned messaging'
        start_char = source_text.index(phrase)
        end_char = start_char + len(phrase)
        candidate_import = CandidateAssertionImport(
            profile=ProfileRef(profile_id=profile_id, profile_version=profile_version),
            payload={
                'predicate': 'oc:signals_alignment',
                'roles': {
                    'subject': [
                        {'kind': 'value', 'value_kind': 'string', 'value': 'Campaign Alpha'}
                    ],
                    'object': [
                        {'kind': 'value', 'value_kind': 'string', 'value': phrase}
                    ],
                },
            },
            submitted_by=submitted_by,
            source_artifact=SourceArtifactRef(
                source_kind=source_kind,
                source_ref=source_ref,
                source_label=source_label,
                source_metadata={},
                content_text=source_text,
            ),
            evidence_spans=(EvidenceSpan(start_char=start_char, end_char=end_char, text=phrase),),
            claim_text='Campaign Alpha used aligned messaging.',
        )
        return (self._review_service.submit_candidate_import(candidate_import=candidate_import),)


In [3]:
workspace = TemporaryDirectory()
workspace_path = Path(workspace.name)
review_db_path = workspace_path / 'review.sqlite3'
overlay_root = workspace_path / 'ontology_overlays'
source_path = workspace_path / 'source.txt'
source_path.write_text('Campaign Alpha used aligned messaging across channels.', encoding='utf-8')

def run_cli(args: list[str]) -> tuple[int, str, str]:
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        exit_code = cli_module.main(args)
    return exit_code, stdout_buffer.getvalue(), stderr_buffer.getvalue()

original_extractor = cli_module.TextExtractionService
cli_module.TextExtractionService = DeterministicTextExtractionService


## Phase 2

Purpose: run the reviewed governance flow entirely through the CLI.

Input -> output: `extracted_candidate_batch -> accepted_review_state`

Acceptance criteria:
1. Candidate review is explicit.
2. Proposal review and overlay application are explicit.
3. No module-level Python calls are needed for the workflow actions.

status: `proven`

execution_mode: `live`

In [4]:
extract_exit, extract_stdout, extract_stderr = run_cli([
    'extract-text',
    '--input', str(source_path),
    '--profile-id', 'psyop_seed',
    '--profile-version', '0.1.0',
    '--submitted-by', 'analyst:notebook',
    '--review-db-path', str(review_db_path),
    '--overlay-root', str(overlay_root),
    '--output', 'json',
])
assert extract_exit == 0, extract_stderr
extract_result = json.loads(extract_stdout)
candidate_id = extract_result[0]['candidate']['candidate_id']
proposal_id = extract_result[0]['proposals'][0]['proposal_id']

review_candidate_exit, _, review_candidate_stderr = run_cli([
    'review-candidate', '--candidate-id', candidate_id, '--decision', 'accepted',
    '--actor-id', 'analyst:reviewer', '--review-db-path', str(review_db_path), '--output', 'json',
])
assert review_candidate_exit == 0, review_candidate_stderr

review_proposal_exit, _, review_proposal_stderr = run_cli([
    'review-proposal', '--proposal-id', proposal_id, '--decision', 'accepted',
    '--actor-id', 'analyst:reviewer', '--acceptance-policy', 'apply_to_overlay',
    '--review-db-path', str(review_db_path), '--output', 'json',
])
assert review_proposal_exit == 0, review_proposal_stderr

apply_exit, _, apply_stderr = run_cli([
    'apply-overlay', '--proposal-id', proposal_id, '--actor-id', 'analyst:reviewer',
    '--review-db-path', str(review_db_path), '--overlay-root', str(overlay_root), '--output', 'json',
])
assert apply_exit == 0, apply_stderr


## Phase 3

Purpose: export the governed bundle artifact that makes the workflow product-facing.

Input -> output: `accepted_review_state -> governed_bundle`

Acceptance criteria:
1. The export succeeds through the CLI.
2. The bundle includes accepted candidates, provenance, and governance state.
3. The exported artifact is inspectable as downstream JSON.

status: `proven`

execution_mode: `live`

In [5]:
export_exit, export_stdout, export_stderr = run_cli([
    'export-governed-bundle',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert export_exit == 0, export_stderr
governed_bundle = json.loads(export_stdout)
assert governed_bundle['summary']['total_candidates'] == 1
assert governed_bundle['summary']['total_linked_proposals'] == 1
assert governed_bundle['candidate_bundles'][0]['candidate']['candidate_id'] == candidate_id
assert governed_bundle['candidate_bundles'][0]['linked_proposals'][0]['proposal_id'] == proposal_id
governed_bundle


{'candidate_bundles': [{'artifact_links': [],
   'artifacts': [],
   'candidate': {'candidate_id': 'cand_fd001216b7654fefb509da8f',
    'claim_text': 'Campaign Alpha used aligned messaging.',
    'evidence_spans': [{'end_char': 37,
      'start_char': 20,
      'text': 'aligned messaging'}],
    'normalized_payload': {'predicate': 'oc:signals_alignment',
     'roles': {'object': [{'kind': 'value',
        'value': 'aligned messaging',
        'value_kind': 'string'}],
      'subject': [{'kind': 'value',
        'value': 'Campaign Alpha',
        'value_kind': 'string'}]}},
    'payload': {'predicate': 'oc:signals_alignment',
     'roles': {'object': [{'kind': 'value',
        'value': 'aligned messaging',
        'value_kind': 'string'}],
      'subject': [{'kind': 'value',
        'value': 'Campaign Alpha',
        'value_kind': 'string'}]}},
    'payload_hash': 'sha256:61808a0228906bbaaf1280423471693fa04fa39e993210adeea8b53d4f7e0b71',
    'profile': {'profile_id': 'psyop_seed', 'prof

In [6]:
cli_module.TextExtractionService = original_extractor
workspace.cleanup()
'cleanup_complete'


'cleanup_complete'